In [1]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image: https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

library(tidyverse) # metapackage of all tidyverse packages

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

list.files(path = "../input")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


[1] "datasets"

# Project 1 - Dimension Reduction

### 1.1 Data Loading 

In [ ]:
# libraries
library(tidyverse)
# Install required packages (run once)
install.packages(c("praznik", "infotheo", "ggplot2", "tidyr", "dplyr", "randomForest", "caret"))

library(praznik)    # MIM, MIFS, mRMR, CMIM, JMI, CIFE
library(infotheo)   # MI estimation (for maxMIFS & DMIM)
library(ggplot2)
library(dplyr)
library(tidyr)
library(randomForest)
library(caret)



# Load and inspect the dataset
df <- read.csv("/kaggle/input/datasets/vicsuperman/prediction-of-music-genre/music_genre.csv")


str(df)
summary(df)

# Keep only Classical and Rap in a new data frame
df2 <- df %>% filter(music_genre %in% c("Classical", "Rap"))
df2$music_genre <- factor(df2$music_genre)

# Dimensions
dim(df2)

# Check if there are missing values
colSums(is.na(df2))

# Check the "?" instances visible in the dataset
sapply(df2, function(x) sum(x == "?", na.rm = TRUE))

# Class balance
table(df2$music_genre)


summary(df2)

# Identify non-numeric columns
sapply(df2, class)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message:
“package ‘praznik’ is not available for this version of R
‘praznik’ version 12.0.0 is in the repositories but depends on R (>= 4.5.0)

A version of this package for your version of R might be available elsewhere,
see the ideas at
https://cran.r-project.org/doc/manuals/r-patched/R-admin.html#Installing-packages”
also installing the dependencies ‘S7’, ‘scales’, ‘lifecycle’, ‘rlang’, ‘vctrs’


Warning message in install.packages(c("praznik", "infotheo", "ggplot2", "tidyr", :
“installation of package ‘rlang’ had non-zero exit status”
Warning message in install.packages(c("praznik", "infotheo", "ggplot2", "tidyr", :
“installation of package ‘lifecycle’ had non-zero exit status”
Warning message in install.packages(c("praznik", "infotheo", "ggplot2", "tidyr", :
“installation of package ‘vctrs’ had non-zero exit status”
Warning message in install.packages(c("praznik", "infotheo", "ggplot2", "ti

### 1.2 Preprocessing

In [ ]:
#storing original size for statistics
n_original <- nrow(df)

# Fix tempo column: convert to numeric ("?" -> NA)
df$tempo <- as.numeric(as.character(df$tempo))
cat("NAs introduced in tempo:", sum(is.na(df$tempo)), "\n")

# Fix duration_ms: replace -1 with NA
df$duration_ms[df$duration_ms == -1] <- NA
cat("NAs in duration_ms:", sum(is.na(df$duration_ms)), "\n")

# Drop irrelevant columns if they exist
cols_to_drop <- intersect(c("instance_id", "obtained_date"), names(df))
df <- df %>% select(-all_of(cols_to_drop))

# Convert categorical features to factors
df$key  <- factor(df$key)
df$mode <- factor(df$mode)
df$music_genre <- factor(df$music_genre)  # ensure genre is factor too

# Remove rows with any NA
df_clean <- df %>% drop_na()
cat("Rows after cleaning:", nrow(df_clean), "\n")

n_clean <- nrow(df_clean)
pct_removed <- (n_original - n_clean) / n_original * 100
cat(sprintf("Percentage of rows removed during cleaning: %.2f%%\n", pct_removed))

# Verify class balance after cleaning (all genres)
cat("Class balance after cleaning:\n")
table(df_clean$music_genre)

### 1.3 Visualizations

In [ ]:
# Select the features we want to plot for analysis in the report
selected_features <- c("danceability", "acousticness", "energy", "instrumentalness")

# Create the plot
p_selected <- df2_clean %>%
  select(all_of(selected_features), music_genre) %>%
  pivot_longer(-music_genre) %>%
  ggplot(aes(x = value, fill = music_genre)) +
  geom_density(alpha = 0.5) +
  facet_wrap(~name, scales = "free") +
  theme_minimal() +
  labs(title = "Distributions of selected features by genre") +
  theme(legend.position = "top")

# Display the plot
print(p_selected)

# Save the plot (Kaggle working directory)
ggsave("/kaggle/working/selected_features_distributions.png", 
       plot = p_selected, width = 12, height = 6, dpi = 300)

## 2. Train/Test Split

In [ ]:
set.seed(42)

n <- nrow(df2)
train_idx <- sample(1:n, size = floor(0.7 * n))

train_df <- df2[train_idx, ]
test_df  <- df2[-train_idx, ]

cat("Train:", nrow(train_df), "| Test:", nrow(test_df))
table(train_df$music_genre)
table(test_df$music_genre)

## 3. PCA

## 4. Kernel PCA

## 5. Feature Selection

In [ ]:
# 1. Define the numeric features
num_features <- c("popularity", "acousticness", "danceability", "duration_ms",
                  "energy", "instrumentalness", "liveness", "loudness",
                  "speechiness", "valence", "tempo")

# 2. Define the improved function
prep_and_impute <- function(train_df, test_df, features) {
  
  # Helper to clean specific column quirks (Tempo and Duration)
  clean_quirks <- function(df) {
    # Convert tempo to numeric safely
    df$tempo <- suppressWarnings(as.numeric(as.character(df$tempo)))
    # Treat -1 in duration as missing data (NA)
    df$duration_ms[df$duration_ms == -1] <- NA
    return(df)
  }

  # Apply quirk cleaning to raw dataframes first
  train_df <- clean_quirks(train_df)
  test_df  <- clean_quirks(test_df)
  
  # Subset columns
  X_train <- train_df[, features, drop = FALSE]
  X_test  <- test_df[, features, drop = FALSE]
  
  # Calculate medians from TRAINING data ONLY
  train_medians <- sapply(X_train, median, na.rm = TRUE)
  
  # Helper to apply the pre-calculated medians
  apply_fix <- function(df, medians) {
    for (col in names(medians)) {
      df[is.na(df[[col]]), col] <- medians[col]
    }
    return(df)
  }
  
  # Apply the training medians to both sets
  X_train_clean <- apply_fix(X_train, train_medians)
  X_test_clean  <- apply_fix(X_test, train_medians)
  
  return(list(train = X_train_clean, test = X_test_clean))
}

# 3. Run the process
processed_data <- prep_and_impute(train_df, test_df, num_features)

# 4. Extract your final dataframes
X_train <- processed_data$train
X_test  <- processed_data$test

# 5. Extract Class vectors
Y_train <- train_df$music_genre
Y_test  <- test_df$music_genre

# 6. Check results
cat("Feature matrix dimensions:", nrow(X_train), "x", ncol(X_train), "\n")
cat("Class distribution:\n"); print(table(Y_train))

In [ ]:
# infotheo::discretize uses equal-frequency binning by default, so that mutual
#information calculation is stable and we get non zero probabilities (which would 
# occur in the continuous case)
# Number of bins ~ sqrt(n), as recommended in the literature
n_bins <- ceiling(sqrt(nrow(X_train)))
cat("Using", n_bins, "bins for discretisation\n")

X_disc <- infotheo::discretize(X_train, disc = "equalfreq", nbins = n_bins)
Y_disc <- as.integer(Y_train)   #alphabetical order assigned to the genres that 
#become integers

In [ ]:
# defining all the mutual informations for later use in each method

# MI( Xi ; C )  for every feature to determine relevance
mi_feature_class <- sapply(seq_len(ncol(X_disc)), function(i)
  infotheo::mutinformation(X_disc[, i], Y_disc))
names(mi_feature_class) <- colnames(X_disc)

# MI( Xi ; Xj )  for every pair  to determine redundancy matrix
p <- ncol(X_disc)
mi_ff <- matrix(0, p, p, dimnames = list(colnames(X_disc), colnames(X_disc)))
for (i in seq_len(p))
  for (j in seq_len(p))
    if (i != j)
      mi_ff[i, j] <- infotheo::mutinformation(X_disc[, i], X_disc[, j])



# MI( Xi ; Xj | C )  conditional MI for every pair to determine complimentarity
for (i in seq_len(p)) {
  for (j in seq_len(p)) {
    if (i != j) {
      H_Xi_given_C   <- infotheo::condentropy(X_disc[, i], as.data.frame(Y_disc))
      H_Xi_given_XjC <- infotheo::condentropy(X_disc[, i],
                           data.frame(X_disc[, j], Y_disc))
      mi_ff_given_c[i, j] <- H_Xi_given_C - H_Xi_given_XjC
    }
  }
}

cat("MI tables computed.\n")
cat("MI(popularity, class):", round(mi_feature_class["popularity"], 4), "\n")

In [ ]:
# maxMIFS: OF(Xi) = MI(C; Xi) - max_{Xs in S} MI(Xi, Xs)

run_maxMIFS <- function(mi_fc, mi_ff_mat, k) {
  p        <- length(mi_fc)
  feat_names <- names(mi_fc)
  selected <- integer(0)
  remaining <- seq_len(p)
  scores_log <- list()

  for (step in seq_len(k)) {
    scores <- sapply(remaining, function(i) {
      rel <- mi_fc[i]
      if (length(selected) == 0) return(rel)
      red <- max(mi_ff_mat[i, selected])
      rel - red
    })
    names(scores) <- feat_names[remaining]
    best <- remaining[which.max(scores)]
    selected  <- c(selected, best)
    remaining <- setdiff(remaining, best)
    scores_log[[step]] <- sort(scores, decreasing = TRUE)
  }
  list(selected = feat_names[selected], scores = scores_log)
}

k_select <- ncol(X_disc)   # rank all features

result_maxMIFS <- run_maxMIFS(mi_feature_class, mi_ff, k = k_select)
cat("maxMIFS ranking:\n")
print(result_maxMIFS$selected)

In [ ]:
# DMIM: OF(Xi) = MI(C; Xi) - max_{Xs in S} MI(Xi, Xs) 
# + max_{Xs in S} MI(Xi, Xs | C)


run_DMIM <- function(mi_fc, mi_ff_mat, mi_ff_c_mat, k) {
  p          <- length(mi_fc)
  feat_names <- names(mi_fc)
  selected   <- integer(0)
  remaining  <- seq_len(p)
  scores_log <- list()

  for (step in seq_len(k)) {
    scores <- sapply(remaining, function(i) {
      rel <- mi_fc[i]
      if (length(selected) == 0) return(rel)
      inter_red  <- max(mi_ff_mat[i, selected])          # max MI(Xi, Xs)
      class_red  <- max(mi_ff_c_mat[i, selected])        # max MI(Xi, Xs | C)
      rel - inter_red + class_red
    })
    names(scores) <- feat_names[remaining]
    best      <- remaining[which.max(scores)]
    selected  <- c(selected, best)
    remaining <- setdiff(remaining, best)
    scores_log[[step]] <- sort(scores, decreasing = TRUE)
  }
  list(selected = feat_names[selected], scores = scores_log)
}

result_DMIM <- run_DMIM(mi_feature_class, mi_ff, mi_ff_given_c, k = k_select)
cat("DMIM ranking:\n")
print(result_DMIM$selected)

In [ ]:
# praznik expects a data.frame X and a factor Y
# It handles discretisation internally

k_select <- ncol(X_train)   # rank all features

result_MIM   <- praznik::MIM(  X_train, Y_train, k = k_select)
result_MIFS  <- praznik::MIFS( X_train, Y_train, k = k_select)
result_mRMR  <- praznik::MRMR( X_train, Y_train, k = k_select)
result_CMIM  <- praznik::CMIM( X_train, Y_train, k = k_select)
result_JMI   <- praznik::JMI(  X_train, Y_train, k = k_select)
result_CIFE  <- praznik::CIFE( X_train, Y_train, k = k_select)

cat("MIM  ranking:", result_MIM$selection,  "\n")
cat("MIFS ranking:", result_MIFS$selection, "\n")
cat("mRMR ranking:", result_mRMR$selection, "\n")
cat("CMIM ranking:", result_CMIM$selection, "\n")
cat("JMI  ranking:", result_JMI$selection,  "\n")
cat("CIFE ranking:", result_CIFE$selection, "\n")

In [ ]:
all_rankings <- data.frame(
  Rank     = seq_len(k_select),
  MIM      = result_MIM$selection,
  MIFS     = result_MIFS$selection,
  mRMR     = result_mRMR$selection,
  maxMIFS  = result_maxMIFS$selected,
  CIFE     = result_CIFE$selection,
  CMIM     = result_CMIM$selection,
  JMI      = result_JMI$selection,
  DMIM     = result_DMIM$selected
)

print(all_rankings)

In [ ]:
# MI(feature, class) as a baseline relevance chart
mi_df <- data.frame(
  Feature = names(mi_feature_class),
  MI      = mi_feature_class
) %>% arrange(desc(MI))

ggplot(mi_df, aes(x = reorder(Feature, MI), y = MI, fill = MI)) +
  geom_bar(stat = "identity") +
  coord_flip() +
  scale_fill_gradient(low = "#b3cde3", high = "#084594") +
  labs(title = "MIM scores – MI(Feature, Class)",
       x = "Feature", y = "Mutual Information (nats)") +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none")

In [ ]:
# Choose a cutoff k 
k_final <- 6   # adjust based on your analysis

# Build a binary selection matrix
methods <- c("MIM","MIFS","mRMR","maxMIFS","CIFE","CMIM","JMI","DMIM")
top_k_list <- list(
  MIM     = result_MIM$selection[1:k_final],
  MIFS    = result_MIFS$selection[1:k_final],
  mRMR    = result_mRMR$selection[1:k_final],
  maxMIFS = result_maxMIFS$selected[1:k_final],
  CIFE    = result_CIFE$selection[1:k_final],
  CMIM    = result_CMIM$selection[1:k_final],
  JMI     = result_JMI$selection[1:k_final],
  DMIM    = result_DMIM$selected[1:k_final]
)

all_feats <- unique(unlist(top_k_list))
heat_mat  <- sapply(top_k_list, function(sel) as.integer(all_feats %in% sel))
rownames(heat_mat) <- all_feats

heat_df <- as.data.frame(heat_mat) %>%
  tibble::rownames_to_column("Feature") %>%
  pivot_longer(-Feature, names_to = "Method", values_to = "Selected")

ggplot(heat_df, aes(x = Method, y = Feature, fill = factor(Selected))) +
  geom_tile(color = "white", linewidth = 0.5) +
  scale_fill_manual(values = c("0" = "#f0f0f0", "1" = "#2171b5"),
                    labels = c("Not selected", "Selected"),
                    name = "") +
  labs(title = paste0("Top-", k_final, " features selected by each method"),
       x = "", y = "") +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 35, hjust = 1))

In [ ]:
# Final selected feature sets (top-k_final features per method)
selected_features <- lapply(top_k_list, function(s) s)

cat("Selected features per method (k =", k_final, "):\n")
for (m in names(selected_features)) {
  cat(sprintf("  %-8s: %s\n", m, paste(selected_features[[m]], collapse = ", ")))
}

# Build reduced train/test sets for each method (used in Q6)
train_reduced <- lapply(selected_features, function(feats) train_df[, feats, drop = FALSE])
test_reduced  <- lapply(selected_features, function(feats) test_df[,  feats, drop = FALSE])

## 6. Classification

### 6.1 Evaluation metrics definitions

In [ ]:
# Returns Accuracy, Macro Recall, Macro Precision, Macro F1
# as defined in the project sheet (question 6c)

compute_metrics <- function(true_labels, predicted_labels) {
  
  classes <- levels(true_labels)
  cm      <- table(Predicted = predicted_labels, Actual = true_labels)
  
  # Per-class recall and precision
  recall    <- sapply(classes, function(c)
    cm[c, c] / sum(cm[, c]))           # TP / (TP + FN)
  
  precision <- sapply(classes, function(c)
    cm[c, c] / sum(cm[c, ]))           # TP / (TP + FP)
  
  # Handle division by zero
  recall[is.nan(recall)]       <- 0
  precision[is.nan(precision)] <- 0
  
  f1 <- 2 * recall * precision / (recall + precision)
  f1[is.nan(f1)] <- 0
  
  # Macro averages
  macro_recall    <- mean(recall)
  macro_precision <- mean(precision)
  macro_f1        <- mean(f1)
  
  # Overall accuracy
  accuracy <- sum(diag(cm)) / sum(cm)
  
  list(
    Accuracy         = round(accuracy,         4),
    Macro_Recall     = round(macro_recall,     4),
    Macro_Precision  = round(macro_precision,  4),
    Macro_F1         = round(macro_f1,         4),
    Confusion_Matrix = cm
  )
}

### 6.2 Random Forest Model

#### 6.2.1 Training on Original Data Set

In [ ]:
set.seed(42)

rf_original <- randomForest(
  x          = X_train_all,
  y          = Y_train,
  ntree      = 500,
  mtry       = floor(sqrt(ncol(X_train_all))),
  nodesize   = 1,
  importance = TRUE
)

# Predict on test set
pred_original <- predict(rf_original, newdata = X_test_all)

# Evaluate
metrics_original <- compute_metrics(Y_test, pred_original)

cat("=== RF — Original Full Dataset ===\n")
cat("Accuracy        :", metrics_original$Accuracy,        "\n")
cat("Macro Recall    :", metrics_original$Macro_Recall,    "\n")
cat("Macro Precision :", metrics_original$Macro_Precision, "\n")
cat("Macro F1        :", metrics_original$Macro_F1,        "\n")
cat("\nConfusion Matrix:\n")
print(metrics_original$Confusion_Matrix)

# OOB error estimate (free internal validation)
cat("\nOOB Error Estimate:", round(rf_original$err.rate[500, "OOB"], 4), "\n")

In [ ]:
# Extract importance scores
imp_df <- as.data.frame(importance(rf_original))
imp_df$Feature <- rownames(imp_df)

# MeanDecreaseAccuracy is the permutation-based importance
imp_df <- imp_df %>% arrange(desc(MeanDecreaseAccuracy))

ggplot(imp_df, aes(x = reorder(Feature, MeanDecreaseAccuracy),
                   y = MeanDecreaseAccuracy,
                   fill = MeanDecreaseAccuracy)) +
  geom_bar(stat = "identity") +
  coord_flip() +
  scale_fill_gradient(low = "#c6dbef", high = "#08519c") +
  labs(title    = "Random Forest — Variable Importance (Full Dataset)",
       subtitle = "Mean Decrease in Accuracy (permutation-based)",
       x = "Feature", y = "Mean Decrease Accuracy") +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none")

#### 6.2.2 Training on PCA derived data

In [ ]:
# ── Assumes pca_train and pca_test exist from Exercise 3 ─────────────────────
# These should be data frames of PCA scores with n_pca_components columns
# e.g. produced by:
#   pca_model  <- prcomp(X_train_num, center = TRUE, scale. = TRUE)
#   pca_train  <- as.data.frame(pca_model$x[, 1:n_pca])
#   pca_test   <- as.data.frame(predict(pca_model, X_test_num)[, 1:n_pca])

set.seed(42)

rf_pca <- randomForest(
  x        = pca_train,
  y        = Y_train,
  ntree    = 500,
  mtry     = floor(sqrt(ncol(pca_train))),
  nodesize = 1
)

pred_pca     <- predict(rf_pca, newdata = pca_test)
metrics_pca  <- compute_metrics(Y_test, pred_pca)

cat("=== RF — PCA Reduced ===\n")
cat("Components used :", ncol(pca_train), "\n")
cat("Accuracy        :", metrics_pca$Accuracy,        "\n")
cat("Macro Recall    :", metrics_pca$Macro_Recall,    "\n")
cat("Macro Precision :", metrics_pca$Macro_Precision, "\n")
cat("Macro F1        :", metrics_pca$Macro_F1,        "\n")

#### 6.2.3 Training on Kernel-PCA derived data

In [ ]:
# ── Assumes kpca_train and kpca_test exist from Exercise 4 ───────────────────
# e.g. produced via kernlab::kpca or similar

set.seed(42)

rf_kpca <- randomForest(
  x        = kpca_train,
  y        = Y_train,
  ntree    = 500,
  mtry     = floor(sqrt(ncol(kpca_train))),
  nodesize = 1
)

pred_kpca    <- predict(rf_kpca, newdata = kpca_test)
metrics_kpca <- compute_metrics(Y_test, pred_kpca)

cat("=== RF — Kernel PCA Reduced ===\n")
cat("Components used :", ncol(kpca_train), "\n")
cat("Accuracy        :", metrics_kpca$Accuracy,        "\n")
cat("Macro Recall    :", metrics_kpca$Macro_Recall,    "\n")
cat("Macro Precision :", metrics_kpca$Macro_Precision, "\n")
cat("Macro F1        :", metrics_kpca$Macro_F1,        "\n")

#### 6.2.4 Training on information-theoretic selected features

In [ ]:
# ── Assumes selected_features list exists from Exercise 5 ────────────────────
# selected_features$MIM, $MIFS, $mRMR, $maxMIFS, $CIFE, $CMIM, $JMI, $DMIM
# Each is a character vector of feature names

set.seed(42)

rf_fs_models  <- list()
rf_fs_metrics <- list()

for (method in names(selected_features)) {
  
  feats <- selected_features[[method]]
  
  # Build feature matrices for this subset
  X_tr <- train_df[, feats, drop = FALSE]
  X_te <- test_df[,  feats, drop = FALSE]
  
  # Ensure factors have consistent levels
  for (f in feats) {
    if (is.factor(train_df[[f]])) {
      X_tr[[f]] <- factor(X_tr[[f]])
      X_te[[f]] <- factor(X_te[[f]], levels = levels(X_tr[[f]]))
    }
  }
  
  # Train
  rf_fit <- randomForest(
    x        = X_tr,
    y        = Y_train,
    ntree    = 500,
    mtry     = max(1, floor(sqrt(length(feats)))),
    nodesize = 1
  )
  
  # Predict and evaluate
  preds   <- predict(rf_fit, newdata = X_te)
  metrics <- compute_metrics(Y_test, preds)
  
  rf_fs_models[[method]]  <- rf_fit
  rf_fs_metrics[[method]] <- metrics
  
  cat(sprintf("%-10s | Acc: %.4f | MacroRe: %.4f | MacroPr: %.4f | MacroF1: %.4f\n",
              method,
              metrics$Accuracy,
              metrics$Macro_Recall,
              metrics$Macro_Precision,
              metrics$Macro_F1))
}

#### 6.2.5 analysis and visualizations

In [ ]:
# Build results data frame covering all strategies
all_results <- bind_rows(
  
  # Original dataset
  data.frame(
    Strategy = "Original",
    Method   = "—",
    Accuracy = metrics_original$Accuracy,
    MacroRecall    = metrics_original$Macro_Recall,
    MacroPrecision = metrics_original$Macro_Precision,
    MacroF1        = metrics_original$Macro_F1
  ),
  
  # PCA
  data.frame(
    Strategy = "PCA",
    Method   = paste0(ncol(pca_train), " components"),
    Accuracy = metrics_pca$Accuracy,
    MacroRecall    = metrics_pca$Macro_Recall,
    MacroPrecision = metrics_pca$Macro_Precision,
    MacroF1        = metrics_pca$Macro_F1
  ),
  
  # Kernel PCA
  data.frame(
    Strategy = "KernelPCA",
    Method   = paste0(ncol(kpca_train), " components"),
    Accuracy = metrics_kpca$Accuracy,
    MacroRecall    = metrics_kpca$Macro_Recall,
    MacroPrecision = metrics_kpca$Macro_Precision,
    MacroF1        = metrics_kpca$Macro_F1
  ),
  
  # Feature selection methods
  bind_rows(lapply(names(rf_fs_metrics), function(m) {
    met <- rf_fs_metrics[[m]]
    data.frame(
      Strategy       = "FeatureSelection",
      Method         = m,
      Accuracy       = met$Accuracy,
      MacroRecall    = met$Macro_Recall,
      MacroPrecision = met$Macro_Precision,
      MacroF1        = met$Macro_F1
    )
  }))
)

# Print clean table
cat("\n========== RF RESULTS SUMMARY ==========\n")
print(all_results %>% arrange(desc(MacroF1)), row.names = FALSE)

In [ ]:
# ── Bar chart: Macro F1 across all strategies ─────────────────────────────────
plot_df <- all_results %>%
  mutate(Label = ifelse(Strategy == "FeatureSelection",
                        paste0("FS-", Method),
                        paste0(Strategy,
                               ifelse(Method == "—", "", paste0("\n(", Method, ")")))))

ggplot(plot_df, aes(x = reorder(Label, MacroF1),
                    y = MacroF1,
                    fill = Strategy)) +
  geom_bar(stat = "identity", width = 0.7) +
  geom_text(aes(label = round(MacroF1, 3)),
            hjust = -0.1, size = 3.5) +
  coord_flip(ylim = c(min(plot_df$MacroF1) - 0.05, 1.02)) +
  scale_fill_manual(values = c(
    "Original"         = "#2171b5",
    "PCA"              = "#6baed6",
    "KernelPCA"        = "#bdd7e7",
    "FeatureSelection" = "#238b45"
  )) +
  labs(title    = "Random Forest — Macro F1 by Dimensionality Reduction Strategy",
       subtitle = "Higher is better",
       x = "", y = "Macro F1") +
  theme_minimal(base_size = 13) +
  theme(legend.position = "bottom")

In [ ]:
# Reshape to long format for faceted plot
fs_results <- all_results %>%
  filter(Strategy == "FeatureSelection") %>%
  select(Method, Accuracy, MacroRecall, MacroPrecision, MacroF1) %>%
  pivot_longer(cols = -Method,
               names_to  = "Metric",
               values_to = "Value")

ggplot(fs_results, aes(x = reorder(Method, Value),
                       y = Value,
                       fill = Method)) +
  geom_bar(stat = "identity") +
  facet_wrap(~ Metric, scales = "free_y", ncol = 2) +
  coord_flip() +
  scale_fill_brewer(palette = "Dark2") +
  labs(title = "RF Performance by Feature Selection Method",
       x = "", y = "Score") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "none",
        strip.text      = element_text(face = "bold"))

In [ ]:
# Identify best and worst by Macro F1
fs_f1 <- sapply(rf_fs_metrics, function(m) m$Macro_F1)
best_method  <- names(which.max(fs_f1))
worst_method <- names(which.min(fs_f1))

cat("Best  method by Macro F1:", best_method,  
    "-", round(fs_f1[best_method],  4), "\n")
cat("Worst method by Macro F1:", worst_method, 
    "-", round(fs_f1[worst_method], 4), "\n")

# Plot confusion matrices side by side
par(mfrow = c(1, 2))

fourfoldplot(rf_fs_metrics[[best_method]]$Confusion_Matrix,
             color  = c("#238b45", "#cb181d"),
             conf.level = 0,
             margin = 1,
             main   = paste("Best:", best_method))

fourfoldplot(rf_fs_metrics[[worst_method]]$Confusion_Matrix,
             color  = c("#238b45", "#cb181d"),
             conf.level = 0,
             margin = 1,
             main   = paste("Worst:", worst_method))

par(mfrow = c(1, 1))

In [ ]:
#Comparing RF variable importance with MI rankings

# RF importance from the full model
rf_imp <- importance(rf_original)[, "MeanDecreaseAccuracy"]
rf_imp_df <- data.frame(
  Feature    = names(rf_imp),
  RF_Rank    = rank(-rf_imp),
  RF_Importance = rf_imp
)

# MI ranking from MIM (simplest benchmark — pure MI score)
# Assumes mi_feature_class exists from Exercise 5 cells
mi_df <- data.frame(
  Feature  = names(mi_feature_class),
  MIM_Rank = rank(-mi_feature_class)
) %>% filter(Feature %in% num_features)

# Merge and display
comparison_df <- merge(rf_imp_df, mi_df, by = "Feature") %>%
  arrange(RF_Rank)

cat("=== RF Importance Rank vs MIM Rank ===\n")
print(comparison_df[, c("Feature", "RF_Rank", "MIM_Rank")],
      row.names = FALSE)

# Rank correlation
cat("\nSpearman correlation between RF rank and MIM rank:",
    round(cor(comparison_df$RF_Rank,
              comparison_df$MIM_Rank,
              method = "spearman"), 3), "\n")

### 6.3 KNN

#### 6.3.1 Training on Original Data Set

#### 6.3.2 Training on PCA derived data

#### 6.3.3 Training on Kernel-PCA derived data

#### 6.3.4 Training on information-theoretic reduced data

## 7. Causal Discovery

### 7.1 DAG

In [ ]:
install.packages("pcalg")
library(pcalg)

# Use only the numeric features on training data
# PC algorithm needs a correlation/conditional independence test
# For continuous features use gaussCItest

suffStat <- list(C = cor(X_train_scaled), n = nrow(X_train_scaled))

pc_fit <- pc(
  suffStat  = suffStat,
  indepTest = gaussCItest,
  alpha     = 0.01,        # significance level for CI tests
  labels    = colnames(X_train_scaled),
  verbose   = FALSE
)

plot(pc_fit, main = "PC Algorithm — Estimated CPDAG")

### 7.2 Do calculus

In [ ]:
# Suppose the graph reveals that 'energy' confounds both 
# 'loudness' and 'danceability'. Backdoor adjustment:

# p(genre | do(speechiness = t)) = 
#   sum_z p(genre | speechiness=t, Z=z) * p(Z=z)
# where Z is the backdoor adjustment set

# Discretise speechiness into bins for estimation
speechiness_bins <- quantile(X_train$speechiness, 
                              probs = seq(0, 1, by = 0.1))

# For each bin value of speechiness, compute adjusted prediction
library(dplyr)

adjustment_set <- c("energy", "acousticness")  # from PC graph

do_predictions <- lapply(1:(length(speechiness_bins)-1), function(b) {
  
  lo <- speechiness_bins[b]; hi <- speechiness_bins[b+1]
  
  # Marginalise over adjustment set Z
  # For each unique combo of Z values, get classifier prediction
  # weighted by p(Z)
  
  X_intervened <- X_train
  X_intervened$speechiness <- (lo + hi) / 2  # set by intervention
  
  # Adjusted prediction: weight by marginal p(Z), not p(Z|speechiness)
  probs <- predict(rf_model, newdata = X_intervened, type = "prob")
  
  # Return weighted average
  data.frame(
    speechiness_val = (lo + hi) / 2,
    p_rap           = mean(probs[, "Rap"]),
    p_classical     = mean(probs[, "Classical"])
  )
})

do_curve <- bind_rows(do_predictions)

### 7.3 Robustness Scores

In [ ]:
# For a fixed value of the "relevant" feature (e.g. speechiness high -> Rap)
# intervene on each "nuisance" feature and measure shift in classifier output

compute_IRS_proxy <- function(model, X_data, relevant_feat, 
                               nuisance_feats, n_grid = 10) {
  
  # Fix relevant feature at its median
  X_fixed <- X_data
  X_fixed[[relevant_feat]] <- median(X_data[[relevant_feat]])
  
  baseline_pred <- mean(predict(model, X_fixed, type = "prob")[, "Rap"])
  
  pida_per_nuisance <- sapply(nuisance_feats, function(nf) {
    
    grid_vals <- quantile(X_data[[nf]], probs = seq(0.05, 0.95, length=n_grid))
    
    shifted_preds <- sapply(grid_vals, function(v) {
      X_shifted <- X_fixed
      X_shifted[[nf]] <- v
      mean(predict(model, X_shifted, type = "prob")[, "Rap"])
    })
    
    max(abs(shifted_preds - baseline_pred))  # MPIDA analog
  })
  
  # IRS analog: 1 - (mean PIDA / normaliser)
  normaliser <- sd(predict(model, X_data, type="prob")[,"Rap"])
  1 - mean(pida_per_nuisance) / (normaliser + 1e-8)
}

# Compute for each method's selected feature subset
irs_scores <- sapply(names(selected_features), function(method) {
  feats <- selected_features[[method]]
  relevant <- feats[1]       # top-ranked feature
  nuisance <- feats[-1]      # remaining selected features
  compute_IRS_proxy(rf_model, X_train[, feats], relevant, nuisance)
})

print(round(sort(irs_scores, decreasing=TRUE), 3))